In [ ]:
import os
import requests
import pandas as pd
import time
import datetime
from dotenv import load_dotenv

# 1. Identity & Path Setup
load_dotenv("../../config/.env")
contact_email = os.getenv("CONTACT_EMAIL", "anonymous@example.com")

# Standardized variables added here
SOURCE_PREFIX = "HL7"
PREFIX_MAP_FILE = "../../config/prefix_map.csv"
raw_ingest_file = "../../data/raw/raw_hl7.csv"
master_file = "../../data/raw/metadata_ingest_undeduped.csv"

# 2. HL7 Specific Configuration
# We define the sources and their SSSOM-ready prefixes
HL7_SOURCES = {
    "HL7v3": {
        "url": "http://terminology.hl7.org/CodeSystem-v3-ReligiousAffiliation",
        "name": "HL7 v3 Religious Affiliation",
        "base_uri": "http://terminology.hl7.org/CodeSystem/v3-ReligiousAffiliation"
    },
    "HL7v2": {
        "url": "http://terminology.hl7.org/CodeSystem-v2-0006",
        "name": "HL7 v2 Table 0006",
        "base_uri": "http://terminology.hl7.org/CodeSystem/v2-0006"
    }
}

HEADERS = {
    'User-Agent': f'ReligiousMappingProject/1.0 (Research; mailto:{contact_email})',
    'Accept': 'application/fhir+json'
}

COLUMN_ORDER = [
    "Source_System", "Primary_Label", "CURIE", "Formal_Label", 
    "Hierarchy_Path", "Synonyms", "Description", "Parent_IDs", 
    "Concept_ID", "URI", "Status", "Extraction_Date"
]

# 3. Update Prefix Map for both HL7 versions
def update_prefix_map(prefix, base_uri, source_name):
    new_entry = pd.DataFrame([{'Prefix': prefix, 'Base_URI': base_uri, 'Source_Name': source_name}])
    if os.path.exists(PREFIX_MAP_FILE):
        pmap = pd.read_csv(PREFIX_MAP_FILE)
        if prefix not in pmap['Prefix'].values:
            pmap = pd.concat([pmap, new_entry], ignore_index=True)
            pmap.to_csv(PREFIX_MAP_FILE, index=False)
    else:
        new_entry.to_csv(PREFIX_MAP_FILE, index=False)

for pref, info in HL7_SOURCES.items():
    update_prefix_map(pref, info['base_uri'], info['name'])

print("Cell 1: HL7 environment and Prefix Map initialized for v2 and v3.")

In [ ]:
def process_hl7_item(item, prefix, base_uri, parent_id=None, parent_path=""):
    """
    Recursively parses FHIR CodeSystem concepts. 
    Captures 'Status' for deprecation tracking.
    """
    rows = []
    timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    # Core Identifiers
    code = str(item.get('code', ''))
    display = item.get('display', 'Unknown')
    system = item.get('system', base_uri)
    
    if not code: 
        return []

    # Build Hierarchy Path
    current_path = f"{parent_path} > {display}" if parent_path else display
    
    # 1. Extraction of Raw Definition (Strictly from source)
    description_value = item.get('definition', "")

    # 2. Extraction of Synonyms (From 'designation' block)
    designations = item.get('designation', [])
    synonym_list = []
    for d in designations:
        val = d.get('value')
        if val and val != display:
            synonym_list.append(val)
    synonyms_value = " | ".join(synonym_list) if synonym_list else ""

    # 3. Capture Status (Deprecation) 
    # Logic: Look into properties for 'status' == 'deprecated'
    concept_status = "active"
    properties = item.get('property', [])
    for prop in properties:
        p_code = prop.get('code')
        if p_code == 'status' and prop.get('valueCode') == 'deprecated':
            concept_status = "deprecated"
        elif p_code == 'deprecationDate':
            # Captures the date if provided
            dt = prop.get('valueDateTime', 'unknown date')
            concept_status = f"deprecated ({dt})"

    # Build the SSSOM-aligned row
    row = {
        'Source_System': f"HL7 {prefix}",
        'Primary_Label': display,
        'CURIE': f"{prefix}:{code}",
        'Formal_Label': display,
        'Hierarchy_Path': current_path,
        'Synonyms': synonyms_value,
        'Description': description_value,
        'Parent_IDs': parent_id if parent_id else "",
        'Concept_ID': code,
        'URI': f"{system}#{code}",
        'Extraction_Date': timestamp,
        'Status': concept_status
    }
    rows.append(row)

    # 4. Recursive Walk: Check for nested 'concept' array (CodeSystem style)
    children = item.get('concept', [])
    for sub_item in children:
        rows.extend(process_hl7_item(sub_item, prefix, base_uri, code, current_path))
            
    return rows

# --- Main Execution ---
if os.path.exists(raw_ingest_file): os.remove(raw_ingest_file)

for prefix, info in HL7_SOURCES.items():
    print(f"\n--- Harvesting {info['name']} ({prefix}) ---")
    
    url = f"{info['url']}.json"
    success = False
    
    # The crucial fix: retry up to 3 times to survive server drops
    for attempt in range(3):
        try:
            response = requests.get(url, headers=HEADERS, timeout=15)
            response.raise_for_status()
            data = response.json()
            success = True
            break
        except Exception as e:
            print(f"Attempt {attempt + 1} failed: {e}")
            time.sleep(2)
            
    if not success:
        print(f"Failed to harvest {prefix} after 3 attempts. Skipping.")
        continue
        
    all_rows = []
    
    # CodeSystems store root concepts in a top-level 'concept' array
    root_concepts = data.get('concept', [])
    
    for item in root_concepts:
        all_rows.extend(process_hl7_item(item, prefix, info['base_uri']))
        
    if not all_rows:
        print(f"Warning: No concepts found for {prefix} in 'concept' array.")
        continue

    print(f"Captured {len(all_rows)} concepts.")
    
    # Save to raw_ingest_file with standard COLUMN_ORDER
    df = pd.DataFrame(all_rows)
    
    # Ensure all columns exist (including 'Status')
    for col in COLUMN_ORDER:
        if col not in df.columns:
            df[col] = ""
    
    df = df[COLUMN_ORDER]
    file_exists = os.path.isfile(raw_ingest_file)
    
    df.to_csv(raw_ingest_file, mode='a', index=False, header=not file_exists, encoding='utf-8-sig')

print(f"\nDone! Objective HL7 data appended to {raw_ingest_file}")

In [ ]:
if os.path.exists(raw_ingest_file):
    df_new = pd.read_csv(raw_ingest_file)
    
    if os.path.exists(master_file):
        df_master = pd.read_csv(master_file)
        
        # Deduplication Check based on URI
        existing_uris = set(df_master['URI'].dropna().unique())
        df_to_add = df_new[~df_new['URI'].isin(existing_uris)]
        
        if not df_to_add.empty:
            df_final = pd.concat([df_master, df_to_add], ignore_index=True)
            df_final.to_csv(master_file, index=False, encoding='utf-8-sig')
            print(f"Added {len(df_to_add)} new {SOURCE_PREFIX} records to master.")
        else:
            print(f"No new {SOURCE_PREFIX} records found to add.")
    else:
        df_new.to_csv(master_file, index=False, encoding='utf-8-sig')
        print(f"Master file created with {len(df_new)} {SOURCE_PREFIX} records.")
else:
    print(f"Error: {raw_ingest_file} not found. Please run the ingest sequence first.")